## Testing the RAG

In [19]:
from dotenv import load_dotenv
from rich import print

load_dotenv()

True

In [20]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./data/zalando_cleaned.pdf"
loader = PyPDFLoader(file_path)

documents = loader.load()

In [21]:
print(len(documents))

52

In [22]:
print(documents[0])

Document(
    metadata={
        'producer': 'Microsoft® Word LTSC',
        'creator': 'Microsoft® Word LTSC',
        'creationdate': '2026-03-17T15:54:52+00:00',
        'author': 'python-docx',
        'moddate': '2026-03-17T15:54:52+00:00',
        'source': './data/zalando_cleaned.pdf',
        'total_pages': 52,
        'page': 0,
        'page_label': '1'
    },
    page_content='how do i create a zalando account? \nto create a zalando account simply follow these steps and 
you\'ll soon be ready to order. \nclick the login icon which can be found on the top right-hand corner of your 
screen. \nenter your email address under \'i\'m new here\' and click on \'register\'. then complete all \nmandatory
fields. \nonce you\'ve completed the registration, you will receive a confirmation email. we suggest \nthat you 
check the information in this email carefully, to make sure that it is correct. \nto confirm your registration, 
click on the link provided in the email. once your registration \nhas been confirmed, you can login and start 
shopping. \nif you would like to place an order from a different country, simply go to that country\'s \nversion of
our site using the correct domain and log in with your existing username and \npassword. learn more about where 
zalando is available. if you\'re using our mobile app, you \ncan change the country by going to "your account" and 
then "app settings". \nhow do i reset my password? \nto reset your password, simply click here. enter the email 
address you use for your zalando \naccount and click send. you\'ll receive an email with a link to reset your 
password. click on \nthe link in the email and you will be prompted to enter a new password. \nyou can then use 
your new password to login to your zalando account. \nplease note: \nthis email may take a couple of hours to reach
you and could appear in your spam or junk \nfolder. \nhow do i update my account information? \nyou can update your
account information by following the steps below: \ndelivery address: you can change your delivery and billing 
address under \'addresses\' in \nyour account. \ncredit card: you can update your preferred credit card during the 
checkout process. \npersonal data (email address, password, contact data): you can change your personal data \nsuch
as your email address, password or contact data under \'personal details\' in my \naccount. \nfashion preferences /
items you like: you can change your fashion preference and items you \nlike under \'recommendation preferences\' in
your account. \nif you\'d like to the change the language on our website, you can do so by clicking on the \nglobe 
symbol in the upper right hand corner. if you\'re using our mobile app, you can change \nthe language by going to 
"your account", and then "app settings".'
)

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)  # noqa: E501

all_splits = text_splitter.split_documents(documents)


In [24]:
print(len(all_splits))

1357

In [25]:
print(all_splits[0])

Document(
    metadata={
        'producer': 'Microsoft® Word LTSC',
        'creator': 'Microsoft® Word LTSC',
        'creationdate': '2026-03-17T15:54:52+00:00',
        'author': 'python-docx',
        'moddate': '2026-03-17T15:54:52+00:00',
        'source': './data/zalando_cleaned.pdf',
        'total_pages': 52,
        'page': 0,
        'page_label': '1'
    },
    page_content='how do i create a zalando account?'
)

In [26]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)
id=vector_store.add_documents(documents=all_splits)

In [27]:
search_queries =['How do I delete my user account?', 'delete user account', 'remove user profile']
pages_to_include = set()

for search in search_queries:
    chunks = vector_store.similarity_search_with_score(search, k=2)

    doc_lookup = {doc.metadata["page"]: doc.page_content for doc in documents}

    min_page = min(doc_lookup.keys(), default=0)
    max_page = max(doc_lookup.keys(), default=0)    

    for chunk in chunks:
        print(f"chunk: {chunk}")
        page = chunk[0].metadata['page']

        if page in doc_lookup:
            neighbors = {p for p in {page-1, page, page+1} if min_page <= p <= max_page and p in doc_lookup}  # noqa: E501
            pages_to_include.update(neighbors)
            print(f"page to included:{pages_to_include}")


    docs = [(p, doc_lookup[p]) for p in sorted(pages_to_include)]
    
    
print(f"docs:{docs}")

chunk: (Document(id='7dce1ba3-2264-4949-9392-d22e4cee15e2', metadata={'producer': 'Microsoft® Word LTSC', 
'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-03-17T15:54:52+00:00', 'author': 'python-docx', 'moddate':
'2026-03-17T15:54:52+00:00', 'source': './data/zalando_cleaned.pdf', 'total_pages': 52, 'page': 1, 'page_label': 
'2'}, page_content="how do i delete my user account? \nyou can delete your user account under 'request or delete 
data' in your account."), 0.7653260872990806)

page to included:{0, 1, 2}

chunk: (Document(id='ea15dc5a-d6f2-43f6-9d36-0507f678e44c', metadata={'producer': 'Microsoft® Word LTSC', 
'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-03-17T15:54:52+00:00', 'author': 'python-docx', 'moddate':
'2026-03-17T15:54:52+00:00', 'source': './data/zalando_cleaned.pdf', 'total_pages': 52, 'page': 29, 'page_label': 
'30'}, page_content='how do i unlink my nike membership?'), 0.4265640087370831)

page to included:{0, 1, 2, 28, 29, 30}

chunk: (Document(id='7dce1ba3-2264-4949-9392-d22e4cee15e2', metadata={'producer': 'Microsoft® Word LTSC', 
'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-03-17T15:54:52+00:00', 'author': 'python-docx', 'moddate':
'2026-03-17T15:54:52+00:00', 'source': './data/zalando_cleaned.pdf', 'total_pages': 52, 'page': 1, 'page_label': 
'2'}, page_content="how do i delete my user account? \nyou can delete your user account under 'request or delete 
data' in your account."), 0.6493977513999929)

...

chunk: (Document(id='7dce1ba3-2264-4949-9392-d22e4cee15e2', metadata={'producer': 'Microsoft® Word LTSC', 
'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-03-17T15:54:52+00:00', 'author': 'python-docx', 'moddate':
'2026-03-17T15:54:52+00:00', 'source': './data/zalando_cleaned.pdf', 'total_pages': 52, 'page': 1, 'page_label': 
'2'}, page_content="how do i delete my user account? \nyou can delete your user account under 'request or delete 
data' in your account."), 0.49687946265993094)

page to included:{0, 1, 2, 3, 4, 28, 29, 30}

chunk: (Document(id='4ff9e1d2-a3ae-4e3f-8840-b1fe0a388092', metadata={'producer': 'Microsoft® Word LTSC', 
'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-03-17T15:54:52+00:00', 'author': 'python-docx', 'moddate':
'2026-03-17T15:54:52+00:00', 'source': './data/zalando_cleaned.pdf', 'total_pages': 52, 'page': 3, 'page_label': 
'4'}, page_content='one of the contact buttons below so that we can deactivate the account and prevent further'), 
0.3899935402565683)

page to included:{0, 1, 2, 3, 4, 28, 29, 30}

docs:[(0, 'how do i create a zalando account? \nto create a zalando account simply follow these steps and you\'ll 
soon be ready to order. \nclick the login icon which can be found on the top right-hand corner of your screen. 
\nenter your email address under \'i\'m new here\' and click on \'register\'. then complete all \nmandatory fields.
\nonce you\'ve completed the registration, you will receive a confirmation email. we suggest \nthat you check the 
information in this email carefully, to make sure that it is correct. \nto confirm your registration, click on the 
link provided in the email. once your registration \nhas been confirmed, you can login and start shopping. \nif you
would like to place an order from a different country, simply go to that country\'s \nversion of our site using the
correct domain and log in with your existing username and \npassword. learn more about where zalando is available. 
if you\'re using our mobile app, you \ncan change the country by going to "your account" and then "app settings". 
\nhow do i reset my password? \nto reset your password, simply click here. enter the email address you use for your
zalando \naccount and click send. you\'ll receive an email with a link to reset your password. click on \nthe link 
in the email and you will be prompted to enter a new password. \nyou can then use your new password to login to 
your zalando account. \nplease note: \nthis email may take a couple of hours to reach you and could appear in your 
spam or junk \nfolder. \nhow do i update my account information? \nyou can update your account information by 
following the steps below: \ndelivery address: you can change your delivery and billing address under \'addresses\'
in \nyour account. \ncredit card: you can update your preferred credit card during the checkout process. \npersonal
data (email address, password, contact data): you can change your personal data \nsuch as your email address, 
password or contact data under \'personal details\' in my \naccount. \nfashion preferences / items you like: you 
can change your fashion preference and items you \nlike under \'recommendation preferences\' in your account. \nif 
you\'d like to the change the language on our website, you can do so by clicking on the \nglobe symbol in the upper
right hand corner. if you\'re using our mobile app, you can change \nthe language by going to "your account", and 
then "app settings".'), (1, "how do i request my data? \nrequesting your data \nif you would like to see your order
history, the saved addresses for your orders, or view or \nadjust your liked items, please go to your account page.
you will also be able to check and \nadjust the information that is saved for you favourite brands or your 
preferred sizes, and \nmanage newsletter or mailing subscriptions in this section of the website. \nif you would 
like to request a downloadable file of your account data that includes contact \ninformation, order history, liked 
items and an overview of when and how you may have \ncontacted us, you can do so in your account. once the file is 
ready, we will send it to you via \nemail. this can take up to one month but usually you should receive your data 
earlier. \nwhat information is in my data report? \nyour data report will contain an overview of your account and 
log-in information, order \noverview, personalisation preferences, and liked items. it also includes information 
about \ndata retention periods and third-party sharing. \nwhat if my data file failed to download? \nif you 
requested your file and it fails to download, it could be because the link has expired. \nthe download link is only
valid for 30 days after it is received. after this, you will need to \nsubmit a new request. \nif your download 
link is valid but failed to download, it could be due to a network issue, a \nproblem with your browser, or an 
issue on the server side. in such cases, check your \ninternet connection or try opening it using a differe